# A0 — Prescribed Autonomy

**The simplest AWP level: a static, single-agent DAG with no dynamic behavior.**

At A0, everything is fixed at design time:
- One or more agents, wired into a **static DAG** (Directed Acyclic Graph)
- No dynamic spawning, no runtime decisions, no tool creation
- The workflow YAML **is** the execution plan — the engine just follows it

### When to use A0
- Simple, predictable pipelines (summarize, translate, classify)
- Tasks where the steps are known ahead of time
- Maximum debuggability — you can trace every input/output

### What distinguishes A0 from higher levels
| Property | A0 | A1+ |
|----------|----|----- |
| Agent count | 1+ (often 1) | 2+ with dependencies |
| Execution | Static, predetermined | Conditional, branching |
| State sharing | Minimal | Explicit `share_output` chains |
| Budget controls | Not required | Required at A2+ |
| Engine | `dag` | `dag` (A1) or `delegation_loop` (A2+) |

## 1. Provider Setup

Choose your LLM provider. Supported: **Ollama** (local), **OpenRouter** (cloud), **Custom** (any OpenAI-compatible API).

In [ ]:
# ============================================================
# Provider Selection — choose ONE of: "ollama", "openrouter", "custom"
# ============================================================
PROVIDER = "openrouter"  # <-- change this

# --- Ollama (local) -------------------------------------------
OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# --- OpenRouter (cloud) ---------------------------------------
OPENROUTER_API_KEY = ""  # paste your key or set env var
OPENROUTER_MODEL = "openai/gpt-5-mini"

# --- Custom OpenAI-compatible API -----------------------------
CUSTOM_API_KEY = ""
CUSTOM_BASE_URL = ""
CUSTOM_MODEL = ""

# ==============================================================
import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key or not url:
        raise ValueError("Set CUSTOM_API_KEY and CUSTOM_BASE_URL")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'")

os.environ["LLM_MODEL"] = MODEL
print(f"Provider: {PROVIDER}  |  Model: {MODEL}")

## 2. The A0 Workflow — YAML Anatomy

An A0 workflow is the **minimal valid AWP manifest**. Let's inspect the `01-hello-world` example:

```yaml
awp: "1.0.0"                    # Protocol version (required)

workflow:
  name: hello-world              # Unique name
  version: "1.0.0"
  description: "Single agent greeting — A0 Prescribed"

orchestration:
  execution:
    mode: sequential             # Only option at A0
  graph:                         # Static DAG — one node
    - id: greeter
      agent: greeter
      depends_on: []             # No dependencies (root node)
      share_output:
        - greeting

state:
  model: shared_dict
  sharing:
    strategy: full
```

**Key observations:**
- One agent (`greeter`), no dependencies
- No `delegation_loop` section — pure DAG execution
- No budget constraints — not required at A0
- The engine reads the graph, runs the agent, collects the output. Done.

## 3. Load, Validate, and Inspect

In [ ]:
from pathlib import Path
from awp.parser import parse_manifest, parse_agent
from awp.validator import validate_graph, check_compliance

PROJECT = Path("/home/shumway/projects/agent-workflow-protocol")
workflow_dir = PROJECT / "examples" / "01-hello-world"

# Parse the YAML manifest
manifest = parse_manifest(workflow_dir / "workflow.awp.yaml")

print(f"Workflow:    {manifest.workflow.name}")
print(f"Version:     {manifest.workflow.version}")
print(f"Description: {manifest.workflow.description}")
print(f"Tags:        {manifest.workflow.tags}")
print()

# Validate the graph structure (rules R1-R32)
graph_result = validate_graph(manifest.orchestration)
print(f"Graph valid: {graph_result.valid}")
print(f"Nodes:       {[n.id for n in manifest.orchestration.graph]}")
print()

# Check autonomy compliance
agents = {}
for agent_dir in sorted((workflow_dir / "agents").iterdir()):
    agent_yaml = agent_dir / "agent.awp.yaml"
    if agent_yaml.exists():
        agent = parse_agent(agent_yaml)
        agents[agent.identity.id] = agent
        print(f"Agent '{agent.identity.id}': role={agent.identity.role}")

compliance = check_compliance(manifest, agents, workflow_path=workflow_dir)
print(f"\nAutonomy level: {compliance.level.name}")

## 4. Execute the A0 Workflow

The DAG engine runs the single agent and returns its output.
Notice how simple this is — no loops, no manager, no budget tracking.

In [ ]:
import json
import logging

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

from awp.runtime import WorkflowRunner

runner = WorkflowRunner(workflow_dir)
result = runner.run("Greet the AWP community and explain what AWP stands for")

print("\n" + "=" * 60)
print("A0 RESULT")
print("=" * 60)
print(json.dumps(result, indent=2, default=str))

## 5. What Just Happened?

The execution flow of an A0 workflow is linear and predictable:

```
1. Parse workflow.awp.yaml
2. Topological sort: [greeter]    ← only one node
3. Run greeter(state={"task": "..."})
4. Collect output: {"greeter": {"greeting": "...", "confidence": 0.9}}
5. Done.
```

**No decisions were made at runtime.** The engine simply followed the graph.
This is the defining characteristic of A0: the YAML **is** the execution plan.

### A0 Compliance Checklist
- [x] Valid `workflow.awp.yaml` with `awp` version field
- [x] `workflow` section with `name`, `version`, `description`
- [x] `orchestration.graph` with at least one agent node
- [x] Each agent has an `agent.awp.yaml` with `output.contract`
- [x] Agent returns `{name: {result_dict}}` with `confidence` float (R17)

In [ ]:
# Verify the agent output contract (R17)
greeter_output = result.get("greeter", {})
confidence = greeter_output.get("confidence", 0.0)

print("Agent Output Contract (R17) Check:")
print(f"  Output is dict:       {isinstance(greeter_output, dict)}")
print(f"  Has 'confidence':     {'confidence' in greeter_output}")
print(f"  Confidence value:     {confidence}")
print(f"  Confidence in [0,1]:  {0.0 <= confidence <= 1.0}")
print()
print("Output keys:", list(greeter_output.keys()))

## 6. Comparison: A0 vs Higher Levels

| | A0 (this notebook) | A1 | A2 | A3 | A4 |
|---|---|---|---|---|---|
| **Engine** | DAG | DAG | Delegation Loop | Delegation Loop | Delegation Loop |
| **Agents** | 1+ (static) | 2+ (static) | Dynamic (manager spawns) | Dynamic | Dynamic + recursive |
| **Decisions at runtime** | None | Conditional edges | Manager chooses workers | Manager + tool creation | Self-organizing hierarchy |
| **Budget required** | No | No | Yes | Yes | Yes |
| **Predictability** | 100% deterministic | Mostly deterministic | Non-deterministic | Non-deterministic | Non-deterministic |

**Next:** Open `A1_adaptive.ipynb` to see how adding dependencies and state sharing transforms a static pipeline into an adaptive multi-agent workflow.